# Week 10: Random Variables and Hypothesis Testing

## SCIE1500 — Analytical Methods for Scientists
### Theme: "Making Decisions from Data"

---

## Overview

**Science Context:** Statistical inference, interpreting experimental results, evidence-based conclusions

**Learning Outcomes:** By the end of this week you should be able to:
1. Define discrete and continuous random variables
2. Calculate expected value, variance, and standard deviation
3. Understand the normal distribution and z-scores
4. Set up and conduct hypothesis tests
5. Interpret p-values correctly
6. Understand Type I and Type II errors

**Exam Alignment:** Q6, Q7, Q8, Q9, Q10

## Setup: Import Required Libraries

Run this cell first to import all the Python libraries we'll use throughout this lesson.

In [ ]:
# Standard imports for SCIE1500 notebooks
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Set plotting style
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

print("✓ Libraries loaded successfully!")

---

## 1. Random Variables

### Definition

A **random variable** $X$ assigns a numerical value to each outcome in a sample space.

### Types

- **Discrete:** Takes countable values (e.g., number of heads, defects, customers)
- **Continuous:** Takes any value in an interval (e.g., height, weight, time)

In [ ]:
# Example: Discrete random variable - sum of two dice

# All possible outcomes
outcomes = []
for d1 in range(1, 7):
    for d2 in range(1, 7):
        outcomes.append(d1 + d2)

# Count frequencies
from collections import Counter
freq = Counter(outcomes)
total = len(outcomes)

print("Random Variable X = Sum of Two Dice")
print("="*40)
print(f"Total outcomes: {total}")
print()
print("Probability distribution:")
print(" X    Frequency   P(X)")
print("-"*30)

X_values = sorted(freq.keys())
probs = [freq[x]/total for x in X_values]

for x in X_values:
    print(f"{x:2d}        {freq[x]}        {freq[x]/total:.4f}")

# Visualize
plt.figure(figsize=(10, 5))
plt.bar(X_values, probs, color='steelblue', edgecolor='black')
plt.xlabel('Sum of two dice (X)', fontsize=12)
plt.ylabel('Probability P(X)', fontsize=12)
plt.title('Probability Distribution: Sum of Two Dice', fontsize=14)
plt.xticks(X_values)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

---

## 2. Expected Value (Mean)

### Definition

The **expected value** of a discrete random variable $X$ is:

$$E[X] = \mu = \sum_x x \cdot P(X = x)$$

This is the "long-run average" — what you expect on average over many trials.

### Properties

1. $E[c] = c$ (constant)
2. $E[cX] = c \cdot E[X]$
3. $E[X + Y] = E[X] + E[Y]$

In [ ]:
# Calculate expected value

# E[X] = sum of x * P(x)
E_X = sum(x * freq[x]/total for x in X_values)

print("Expected Value Calculation")
print("="*40)
print("E[X] = Σ x · P(X = x)")
print()
print(" x    P(x)      x·P(x)")
print("-"*30)
for x in X_values:
    p_x = freq[x]/total
    print(f"{x:2d}   {p_x:.4f}    {x * p_x:.4f}")
print("-"*30)
print(f"Sum:           E[X] = {E_X:.4f}")
print()
print("On average, the sum of two dice is 7.")

---

## 3. Variance and Standard Deviation

### Variance

The **variance** measures spread around the mean:

$$\text{Var}(X) = \sigma^2 = E[(X - \mu)^2] = E[X^2] - (E[X])^2$$

### Standard Deviation

$$\sigma = \sqrt{\text{Var}(X)}$$

Standard deviation has the same units as $X$, making it more interpretable.

In [ ]:
# Calculate variance and standard deviation

# Method 1: E[(X - μ)²]
var_method1 = sum((x - E_X)**2 * freq[x]/total for x in X_values)

# Method 2: E[X²] - (E[X])²
E_X2 = sum(x**2 * freq[x]/total for x in X_values)
var_method2 = E_X2 - E_X**2

std_dev = np.sqrt(var_method1)

print("Variance and Standard Deviation")
print("="*45)
print(f"E[X] = {E_X:.4f}")
print(f"E[X²] = {E_X2:.4f}")
print()
print("Method 1: Var(X) = E[(X - μ)²]")
print(f"Var(X) = {var_method1:.4f}")
print()
print("Method 2: Var(X) = E[X²] - (E[X])²")
print(f"Var(X) = {E_X2:.4f} - {E_X:.4f}² = {var_method2:.4f}")
print()
print(f"Standard deviation: σ = √Var(X) = {std_dev:.4f}")

---

## 4. The Normal Distribution

### Definition

The **normal distribution** (Gaussian) is a continuous probability distribution with:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

### Key Properties

- Bell-shaped and symmetric around $\mu$
- Mean = Median = Mode
- Completely determined by $\mu$ and $\sigma$

### The Empirical Rule (68-95-99.7)

- 68% of data within $\mu \pm 1\sigma$
- 95% of data within $\mu \pm 2\sigma$
- 99.7% of data within $\mu \pm 3\sigma$

In [ ]:
# Visualizing the normal distribution

mu = 0
sigma = 1

x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x, mu, sigma)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Normal curve with shading
ax1.plot(x, y, 'b-', linewidth=2)
ax1.fill_between(x[(x >= -1) & (x <= 1)], y[(x >= -1) & (x <= 1)], alpha=0.3, color='blue', label='68%')
ax1.fill_between(x[(x >= -2) & (x < -1)], y[(x >= -2) & (x < -1)], alpha=0.3, color='green')
ax1.fill_between(x[(x > 1) & (x <= 2)], y[(x > 1) & (x <= 2)], alpha=0.3, color='green', label='95%')
ax1.fill_between(x[(x >= -3) & (x < -2)], y[(x >= -3) & (x < -2)], alpha=0.3, color='orange')
ax1.fill_between(x[(x > 2) & (x <= 3)], y[(x > 2) & (x <= 3)], alpha=0.3, color='orange', label='99.7%')

ax1.axvline(x=0, color='red', linestyle='--', alpha=0.7)
ax1.set_xlabel('x (z-score)', fontsize=12)
ax1.set_ylabel('Probability Density', fontsize=12)
ax1.set_title('Standard Normal Distribution N(0, 1)', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Different means and standard deviations
params = [(0, 1, 'blue'), (0, 0.5, 'green'), (2, 1, 'red')]
x2 = np.linspace(-4, 6, 1000)

for mu, sigma, color in params:
    y2 = stats.norm.pdf(x2, mu, sigma)
    ax2.plot(x2, y2, color=color, linewidth=2, label=f'μ={mu}, σ={sigma}')

ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('Probability Density', fontsize=12)
ax2.set_title('Normal Distributions with Different Parameters', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Empirical Rule:")
print(f"P(-1 < Z < 1) = {stats.norm.cdf(1) - stats.norm.cdf(-1):.4f} ≈ 68%")
print(f"P(-2 < Z < 2) = {stats.norm.cdf(2) - stats.norm.cdf(-2):.4f} ≈ 95%")
print(f"P(-3 < Z < 3) = {stats.norm.cdf(3) - stats.norm.cdf(-3):.4f} ≈ 99.7%")

---

## 5. Z-Scores (Standardization)

### Definition

The **z-score** converts any normal variable to the standard normal:

$$z = \frac{x - \mu}{\sigma}$$

### Interpretation

A z-score tells you how many standard deviations $x$ is from the mean.

- $z = 0$: exactly at the mean
- $z = 1$: one standard deviation above
- $z = -2$: two standard deviations below

In [ ]:
# Z-score example

print("Z-Score Example: IQ Scores")
print("="*45)
print("IQ ~ N(μ = 100, σ = 15)")
print()

mu = 100
sigma = 15

# What's the z-score for IQ = 130?
x = 130
z = (x - mu) / sigma
percentile = stats.norm.cdf(z) * 100

print(f"Question: What percentile is an IQ of {x}?")
print()
print(f"z = (x - μ) / σ = ({x} - {mu}) / {sigma} = {z:.2f}")
print(f"P(X < {x}) = P(Z < {z}) = {percentile:.2f}%")
print()
print(f"An IQ of {x} is in the {percentile:.1f}th percentile.")

# Reverse: What IQ corresponds to the 90th percentile?
print("\n" + "-"*45)
print("Question: What IQ is needed for the 90th percentile?")
print()

z_90 = stats.norm.ppf(0.90)  # z-score for 90th percentile
x_90 = mu + z_90 * sigma

print(f"z for 90th percentile = {z_90:.2f}")
print(f"x = μ + z·σ = {mu} + {z_90:.2f}×{sigma} = {x_90:.1f}")
print(f"\nYou need an IQ of about {x_90:.0f} to be in the 90th percentile.")

---

## 6. Introduction to Hypothesis Testing

### The Logic of Hypothesis Testing

1. Assume the null hypothesis $H_0$ is true
2. Calculate the probability of observing data as extreme as ours
3. If this probability (p-value) is very small, reject $H_0$

### Key Terminology

- **Null Hypothesis ($H_0$):** The default claim (usually "no effect" or "no difference")
- **Alternative Hypothesis ($H_a$):** What we're trying to prove
- **Significance Level ($\alpha$):** Threshold for rejection (typically 0.05)
- **P-value:** Probability of getting data as extreme or more, assuming $H_0$ is true

In [ ]:
# Hypothesis testing example: One-sample z-test

print("Hypothesis Testing Example")
print("="*55)
print()
print("A company claims their energy bars contain 200 calories.")
print("A sample of 36 bars has a mean of 208 calories.")
print("Population σ = 15. Is this significantly different from 200?")
print()

# Given
mu_0 = 200    # Claimed mean (null hypothesis)
x_bar = 208   # Sample mean
sigma = 15    # Population std dev
n = 36        # Sample size
alpha = 0.05  # Significance level

print("Step 1: State the hypotheses")
print(f"  H₀: μ = {mu_0} (bars contain 200 calories)")
print(f"  Hₐ: μ ≠ {mu_0} (bars don't contain 200 calories)")
print(f"  This is a two-tailed test at α = {alpha}")
print()

print("Step 2: Calculate the test statistic")
SE = sigma / np.sqrt(n)  # Standard error
z = (x_bar - mu_0) / SE
print(f"  Standard Error = σ/√n = {sigma}/√{n} = {SE:.2f}")
print(f"  z = (x̄ - μ₀) / SE = ({x_bar} - {mu_0}) / {SE:.2f} = {z:.2f}")
print()

print("Step 3: Calculate the p-value")
p_value = 2 * (1 - stats.norm.cdf(abs(z)))  # Two-tailed
print(f"  p-value = 2 × P(Z > |{z:.2f}|) = {p_value:.4f}")
print()

print("Step 4: Make a decision")
if p_value < alpha:
    print(f"  p-value ({p_value:.4f}) < α ({alpha})")
    print("  REJECT H₀: There is evidence the mean is not 200 calories.")
else:
    print(f"  p-value ({p_value:.4f}) ≥ α ({alpha})")
    print("  FAIL TO REJECT H₀: Not enough evidence to reject the claim.")

In [ ]:
# Visualize the hypothesis test

x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x, 0, 1)

# Critical values for two-tailed test at α = 0.05
z_crit = stats.norm.ppf(1 - alpha/2)

plt.figure(figsize=(12, 6))
plt.plot(x, y, 'b-', linewidth=2, label='Standard Normal')

# Shade rejection regions
x_left = x[x <= -z_crit]
x_right = x[x >= z_crit]
plt.fill_between(x_left, stats.norm.pdf(x_left), alpha=0.3, color='red', label=f'Rejection region (α/2 = {alpha/2})')
plt.fill_between(x_right, stats.norm.pdf(x_right), alpha=0.3, color='red')

# Mark the test statistic
plt.axvline(x=z, color='green', linestyle='--', linewidth=2, label=f'Test statistic z = {z:.2f}')
plt.scatter([z], [0], color='green', s=100, zorder=5)

# Critical values
plt.axvline(x=-z_crit, color='red', linestyle=':', linewidth=2)
plt.axvline(x=z_crit, color='red', linestyle=':', linewidth=2, label=f'Critical values ±{z_crit:.2f}')

plt.xlabel('z', fontsize=12)
plt.ylabel('Probability Density', fontsize=12)
plt.title('Two-Tailed Hypothesis Test', fontsize=14)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.annotate(f'z = {z:.2f}\np = {p_value:.4f}', xy=(z, 0.02), xytext=(z+0.5, 0.15),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=11, color='green')

plt.show()

print(f"Since z = {z:.2f} falls in the rejection region (|z| > {z_crit:.2f}), we reject H₀.")

---

## 7. Type I and Type II Errors

### Decision Table

|  | H₀ True | H₀ False |
|--|---------|----------|
| Reject H₀ | **Type I Error** (α) | Correct (Power) |
| Fail to Reject H₀ | Correct | **Type II Error** (β) |

### Definitions

- **Type I Error ($\alpha$):** Rejecting $H_0$ when it's actually true (false positive)
- **Type II Error ($\beta$):** Failing to reject $H_0$ when it's actually false (false negative)
- **Power ($1 - \beta$):** Probability of correctly rejecting a false $H_0$

In [ ]:
# Visualizing Type I and Type II errors

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = np.linspace(-4, 8, 1000)

# Null distribution (H0: μ = 0)
mu_0 = 0
# Alternative (true mean is actually 2)
mu_true = 2

y_null = stats.norm.pdf(x, mu_0, 1)
y_alt = stats.norm.pdf(x, mu_true, 1)

# Critical value for one-tailed test at α = 0.05
z_crit = stats.norm.ppf(0.95, mu_0, 1)

# Left: Type I error (α)
ax1.plot(x, y_null, 'b-', linewidth=2, label='Null distribution (H₀ true)')
ax1.fill_between(x[x >= z_crit], y_null[x >= z_crit], alpha=0.5, color='red', label='Type I Error (α)')
ax1.axvline(x=z_crit, color='black', linestyle='--', linewidth=2)
ax1.set_xlabel('Test Statistic', fontsize=12)
ax1.set_ylabel('Probability Density', fontsize=12)
ax1.set_title('Type I Error: Reject H₀ when H₀ is TRUE', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Type II error (β)
ax2.plot(x, y_null, 'b-', linewidth=2, label='Null (H₀)')
ax2.plot(x, y_alt, 'g-', linewidth=2, label='Alternative (H₀ false)')
ax2.fill_between(x[x < z_crit], y_alt[x < z_crit], alpha=0.5, color='orange', label='Type II Error (β)')
ax2.fill_between(x[x >= z_crit], y_alt[x >= z_crit], alpha=0.3, color='green', label='Power (1-β)')
ax2.axvline(x=z_crit, color='black', linestyle='--', linewidth=2)
ax2.set_xlabel('Test Statistic', fontsize=12)
ax2.set_ylabel('Probability Density', fontsize=12)
ax2.set_title('Type II Error: Fail to Reject H₀ when H₀ is FALSE', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate β and power
beta = stats.norm.cdf(z_crit, mu_true, 1)
power = 1 - beta

print(f"For α = 0.05, critical value = {z_crit:.2f}")
print(f"If true mean = {mu_true}:")
print(f"  Type II Error (β) = {beta:.3f} = {beta*100:.1f}%")
print(f"  Power (1-β) = {power:.3f} = {power*100:.1f}%")

---

## 8. Interpreting P-values Correctly

### What the P-value IS:

> The probability of observing data as extreme or more extreme than what we got, **assuming $H_0$ is true**.

### What the P-value is NOT:

- ❌ The probability that $H_0$ is true
- ❌ The probability that $H_a$ is false
- ❌ The probability of making an error

### Decision Rule

- **p-value < α:** Reject $H_0$ (statistically significant)
- **p-value ≥ α:** Fail to reject $H_0$ (not statistically significant)

In [ ]:
# P-value interpretation practice

print("P-value Interpretation")
print("="*55)
print()

scenarios = [
    (0.001, 0.05, "drug trial shows new drug is more effective"),
    (0.07, 0.05, "teaching method improves test scores"),
    (0.03, 0.01, "diet reduces cholesterol levels"),
    (0.049, 0.05, "advertising increases sales")
]

for p, alpha, context in scenarios:
    result = "REJECT H₀" if p < alpha else "FAIL TO REJECT H₀"
    significant = "Yes" if p < alpha else "No"
    
    print(f"Context: Testing if {context}")
    print(f"  p-value = {p}, α = {alpha}")
    print(f"  Decision: {result}")
    print(f"  Statistically significant? {significant}")
    print()

---

## Key Formulas Summary

| Topic | Key Formula |
|-------|-------------|
| Expected value | $E[X] = \sum x \cdot P(X=x)$ |
| Variance | $\text{Var}(X) = E[X^2] - (E[X])^2$ |
| Standard deviation | $\sigma = \sqrt{\text{Var}(X)}$ |
| Z-score | $z = \frac{x - \mu}{\sigma}$ |
| Standard error | $SE = \frac{\sigma}{\sqrt{n}}$ |
| Z-test statistic | $z = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}}$ |

---

## Self-Assessment Checklist

Before moving to Week 11, ensure you can:

- [ ] Distinguish discrete and continuous random variables
- [ ] Calculate expected value from a probability distribution
- [ ] Calculate variance and standard deviation
- [ ] Describe properties of the normal distribution
- [ ] Convert values to z-scores and back
- [ ] Find probabilities using z-scores
- [ ] State null and alternative hypotheses
- [ ] Calculate p-values for z-tests
- [ ] Make correct decisions based on p-values
- [ ] Explain Type I and Type II errors

---

*Next: Week 11 — Trigonometric Functions*